In [4]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import random
from pathlib import Path
# from tqdm import tqdm
# import json
# from datetime import datetime
# import seaborn as sns
import os
# from ptflops import get_model_complexity_info

RuntimeError: operator torchvision::nms does not exist

In [ ]:
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 2

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Treinando em:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
def load_data_from_folders(dir_data, class_names, reshape_type):
    """Varre as subpastas, lista os caminhos das imagens e associa a um rótulo numérico."""
    data_list = []

    for class_index, class_name in enumerate(class_names):
        dir_class = Path(dir_data) / class_name / reshape_type

        if dir_class.exists():
            images = list(dir_class.glob('*.*'))
            print(f"Imagens encontradas em {class_name}: {len(images)}")

            for img_path in images:
                data_list.append((str(img_path), class_index))
        else:
            print(f"AVISO: Diretório {dir_class} não encontrado!!!!!")

    return data_list

In [ ]:
class MedicalImageDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path, label = self.data[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

In [ ]:
def split_data(data_list, train_size=0.7, inner_train_size=0.8, seed=SEED):
      '''Separa a base de dados de acordo com as especificações pedidas.
      Usando train_test_split em vez de random_split pois as classes são desbalanceadas.'''

      if len(data_list) == 0:
          raise ValueError("data_list vazio")

      paths, labels = zip(*data_list)

      train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
          paths, labels, train_size=train_size, stratify=labels, random_state=seed
      )

      train_paths, val_paths, train_labels, val_labels = train_test_split(
          train_val_paths, train_val_labels, train_size=inner_train_size,
          stratify=train_val_labels, random_state=seed
      )

      # juntando as sublista de paths e labels em uma lista de tuplas (path, labels)
      train_data = list(zip(train_paths, train_labels))
      val_data = list(zip(val_paths, val_labels))
      test_data = list(zip(test_paths, test_labels))

      return train_data, val_data, test_data

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [ ]:
def run_experiment(exp_name, model, train_loader, val_loader, test_loader):
    """Executa um ciclo completo de treinamento, validação e teste para um modelo."""

    # configurações de treinamento
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    print(f'\n--- Treinamento: {exp_name} ---')
    print(f'{"="*60}\n')

    for epoch in range(EPOCHS):
        # --- Fase de Treino ---
        model.train()
        running_loss, correct_preds, total_preds = 0.0, 0, 0

        # train_epoch
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_preds += torch.sum(preds == labels.data)
            total_preds += labels.size(0)

            epoch_train_loss = running_loss / len(train_loader.dataset)
            epoch_train_acc = correct_preds.double() / len(train_loader.dataset)
            history['train_loss'].append(epoch_train_loss)
            history['train_acc'].append(epoch_train_acc.item())

            # --- Fase de Validação ---
            model.eval()
            running_loss, correct_preds, total_preds = 0.0, 0, 0
            all_val_labels, all_val_preds = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    running_loss += loss.item() * inputs.size(0)
                    _, preds = torch.max(outputs, 1)
                    correct_preds += torch.sum(preds == labels.data)
                    total_preds += labels.size(0)

            epoch_val_loss = running_loss / len(val_loader.dataset)
            epoch_val_acc = correct_preds.double() / len(val_loader.dataset)
            history['val_loss'].append(epoch_val_loss)
            history['val_acc'].append(epoch_val_acc.item())

            print(f'Época {epoch+1}/{EPOCHS}: '
                    f'Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | '
                    f'Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}')

    # --- Fase de Teste ---
    print("\n--- Avaliação no Conjunto de Teste ---")
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    # calculando metricas
    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)

    print(f'Acurácia no Teste: {accuracy:.4f}')
    print(f'F1-Score (Macro) no Teste: {f1:.4f}')

    return {'accuracy': accuracy, 'f1_score': f1}

In [ ]:
def plot_f1_comparison(df_results, save_path):
    """Plota comparação de F1-scores"""
    plt.figure(figsize=(16, 6))

    # labels para o eixo x
    df_results['label'] = df_results['model'] + '\n' + df_results['mode'] + '\n' + df_results['aug'].apply(lambda x: 'Aug' if x else 'NoAug')

    colors = plt.cm.viridis(np.linspace(0, 1, len(df_results)))
    bars = plt.bar(range(len(df_results)), df_results['f1_score'], color=colors)

    plt.xlabel('Configuração', fontsize=12)
    plt.ylabel('F1-Score (Macro)', fontsize=12)
    plt.title('Comparação de F1-Score por Configuração', fontsize=14, fontweight='bold')
    plt.xticks(range(len(df_results)), df_results['label'], rotation=45, ha='right', fontsize=8)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
def run_all_experiments(dataset_path, dataset_type, class_names, save_dir_base):
    """Função principal que orquestra todos os experimentos para um dataset."""

    # carregar e dividir o dados
    data_list = load_data_from_folders(dataset_path, class_names, dataset_type)
    train_data, val_data, test_data = split_data(data_list)

    print(f'\nDataset: {dataset_type}')
    print(f'Classes: {class_names}')
    print(f'Divisão: Treino: {len(train_data)}, Validação: {len(val_data)}, Teste: {len(test_data)}')

    # defininfo combinações de experimentos
    all_results = []
    save_dir = Path(save_dir_base) / dataset_type

    # loop principal de experimentos
 
    exp_name = f'{dataset_type}'

    train_dataset = MedicalImageDataset(train_data, transform=transform)
    val_dataset = MedicalImageDataset(val_data, transform=transform)
    test_dataset = MedicalImageDataset(test_data, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    # Criar Modelo
    model =  models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1).to(DEVICE)
    print(f'\nExecutando: {exp_name}')

    # executar cada experimento
    try:
        results = run_experiment(exp_name, model, train_loader, val_loader, test_loader, class_names, save_dir)
        all_results.append({
            'dataset': dataset_type,
            'accuracy': results['accuracy'],
            'f1_score': results['f1_score']
        })
    except Exception as e:
        print(f'ERRO no experimento {exp_name}: {e}')

    # salvar resultados
    df = pd.DataFrame(all_results)
    summary_path = save_dir / f'summary_results_{dataset_type}.csv'
    df.to_csv(summary_path, index=False)
    print(f'\nResultados consolidados salvos em: {summary_path}')

    #  visualizações comparativas
    plot_f1_comparison(df, save_dir / f'f1_comparison_{dataset_type}.png')

    print("\n--- Resumo dos Experimentos ---")
    print(df)
    return df

In [ ]:

f_classic_path = '../feature_to_image/saida'

displasia_classes = ['healthy', 'severe']

save_dir = 'LBP_results'

df_displasia = run_all_experiments(f_classic_path, 'F-Classical', displasia_classes, save_dir)

